# How Large Language Models Work, Explained Simply
Next-token prediction, tokens, temperature and the causal mask, then a tiny GPT trained from scratch in PyTorch that writes little stories. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
2,500 tiny stories (made up for this video), one per line: 178,000 characters. Our model will learn to write more like them.

### 2,500 tiny stories
- `urlretrieve`: download the file into Colab, unless it is already there
- `text[:220]`: the first three stories

In [ ]:
import os, urllib.request

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "how-llms-work/tiny_stories.txt")
if not os.path.exists("tiny_stories.txt"):
    urllib.request.urlretrieve(url, "tiny_stories.txt")
text = open("tiny_stories.txt").read()
print(len(text), "characters")
print(text[:220])

### Characters as tokens
- `chars`: every different character: our tokens
- `encode`: text -> token numbers
- `decode`: token numbers -> text
- `| in the output`: stands for the new-line character

In [ ]:
import torch

chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda ids: "".join(chars[i] for i in ids)
print(len(chars), "tokens:", "".join(chars).replace("\n", "|"))
print(encode("tom found"))
print(decode(encode("tom found")))

## 2. Next-word prediction by hand
A language model predicts what comes next. The simplest way: **count**. P(next | word) = count(word, next) / count(word).

### Counting next words
- `zip(words, words[1:])`: every word paired with the word after it
- `Counter`: how many times each pair appears
- `the output`: after the: cat 2 of 3 times; after cat: sat and ran, half each

In [ ]:
from collections import Counter

tiny = "the cat sat . the cat ran . the dog sat ."
words = tiny.split()
pairs = Counter(zip(words, words[1:]))
for nxt in ["cat", "dog"]:
    print(f"P({nxt} | the) =", round(pairs[("the", nxt)] / words.count("the"), 3))
for nxt in ["sat", "ran"]:
    print(f"P({nxt} | cat) =", round(pairs[("cat", nxt)] / words.count("cat"), 3))

### Temperature by hand
- `scores / temp`: divide the scores by the temperature before softmax
- `0.5`: sharper: the top choice gets even more likely
- `2.0`: flatter: unlikely choices get more of a chance

In [ ]:
scores = torch.tensor([2.0, 1.0, 0.0])
for temp in [1.0, 0.5, 2.0]:
    probs = (scores / temp).softmax(0)
    print(f"temperature {temp}:", [round(p, 3) for p in probs.tolist()])

## 3. Training examples
Cut the text into chunks of 64 characters. The target is the same chunk shifted one character left: at every position, the answer is the next character.

### Inputs and targets
- `T, B = 64, 32`: chunks of 64 characters, 32 chunks per batch
- `d[i + 1:i + T + 1]`: the same chunk, one character later
- `the output`: the target is the input shifted by one

In [ ]:
data = torch.tensor(encode(text))
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
T, B = 64, 32

def get_batch(d):
    starts = torch.randint(len(d) - T - 1, (B,))
    x = torch.stack([d[i:i + T] for i in starts])
    y = torch.stack([d[i + 1:i + T + 1] for i in starts])
    return x, y

torch.manual_seed(0)
x, y = get_batch(train_data)
print("input: ", repr(decode(x[0][:30].tolist())))
print("target:", repr(decode(y[0][:30].tolist())))

## 4. A tiny GPT
Token embeddings + position embeddings, then transformer layers with a **causal mask** (each position may only look at itself and earlier positions), then a layer that gives a score for every possible next character.

### The model
- `nn.TransformerEncoderLayer`: attention + feed-forward + layer norm + residuals
- `layers=2`: two layers stacked
- `generate_square_subsequent_mask`: the causal mask: no peeking at later characters
- `self.out`: a score for each of the 28 possible next characters

In [ ]:
import torch.nn as nn

class TinyGPT(nn.Module):
    def __init__(self, d=64, heads=4, layers=2):
        super().__init__()
        self.tok = nn.Embedding(len(chars), d)
        self.pos = nn.Embedding(T, d)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0,
                                           batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(
            layer, layers, enable_nested_tensor=False)
        self.out = nn.Linear(d, len(chars))
    def forward(self, x):
        t = x.shape[1]
        mask = nn.Transformer.generate_square_subsequent_mask(t)
        h = self.tok(x) + self.pos(torch.arange(t))
        return self.out(self.blocks(h, mask=mask, is_causal=True))

### Build it, and see the mask
- `model = TinyGPT()`: build the model
- `0`: allowed to look
- `-inf`: blocked: after softmax its weight becomes exactly 0

In [ ]:
torch.manual_seed(0)
model = TinyGPT()
print("weights:", sum(p.numel() for p in model.parameters()))
print(nn.Transformer.generate_square_subsequent_mask(5))

### Writing text
- `ids[:, -T:]`: only the last 64 characters fit: the context window
- `[0, -1]`: the scores for the character after the last one
- `torch.multinomial`: pick one character at random, using the probabilities
- `the output`: before training: random characters

In [ ]:
def generate(start, n=150, temp=1.0):
    ids = torch.tensor([encode(start)])
    for _ in range(n):
        with torch.no_grad():
            scores = model(ids[:, -T:])[0, -1]
        probs = (scores / temp).softmax(0)
        nxt = torch.multinomial(probs, 1)
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    return decode(ids[0].tolist())

torch.manual_seed(1)
print(generate("tom found", n=60))

## 5. Train it
Predict the next character at every position; the loss is cross-entropy. About a minute on Colab.

### Train for 1,500 steps
- `AdamW`: Adam with weight decay, the usual choice for transformers
- `.view(-1, len(chars))`: every position of every chunk is one prediction
- `the output`: the loss falls from about 3.5 (guessing) to under 0.3

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)
loss_fn = nn.CrossEntropyLoss()
for step in range(1501):
    x, y = get_batch(train_data)
    loss = loss_fn(model(x).view(-1, len(chars)), y.view(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 300 == 0:
        with torch.no_grad():
            xv, yv = get_batch(val_data)
            val = loss_fn(model(xv).view(-1, len(chars)), yv.view(-1))
        print(f"step {step}: train loss {loss.item():.2f}, validation {val.item():.2f}")

### Write stories at three temperatures
- `0.5`: safe and repetitive
- `1.5`: more surprising, and more mistakes

In [ ]:
for temp in [0.5, 1.0, 1.5]:
    torch.manual_seed(1)
    print(f"--- temperature {temp}")
    print(generate("one day", n=120, temp=temp))

## 6. Mini project: story starters
Give the model the start of a story, and let it finish.

### Finish my story
- `split("\n")[0]`: keep only the first story
- `temp=0.8`: a little safer than 1.0

In [ ]:
torch.manual_seed(7)
for start in ["lily lost", "in the beach,", "ben and zoe"]:
    story = generate(start, n=100, temp=0.8)
    print(story.split("\n")[0])

## Try it yourself
1. Work out the temperature example by hand for temperature 4. Then check it with the code.
2. Train for only 300 steps. How do the stories change?
3. Start the story with your own words (only use letters from the stories).